[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module2_DataSimilarity/03_NLP_SentimentAnalysis.ipynb#copy=true)


# Sentiment Analysis: Supervised Learning with Text

## Learning objectives

By the end of this lesson, you should be able to:

- Formulate sentiment analysis as a binary classification problem.
- Fit a TF–IDF plus logistic-regression pipeline.
- Evaluate a classifier with accuracy, a confusion matrix, and examples of errors.
- Identify ethical and technical limitations of sentiment labels.

**Student Learning Outcome (SLO 4):**
> Train and evaluate a text classifier, then communicate a limitation revealed by its errors.


## A labeled text problem

For sentiment analysis, each review is paired with a label such as positive (1) or negative (0). We will use a deliberately small example so that every prediction is inspectable. Small accuracy values should not be over-interpreted: the goal is to understand the workflow.


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

reviews = np.array([
    "I loved the warm service and flavorful meal", "The plot was thoughtful and moving",
    "Excellent design and easy to use", "This was delightful and worth the price",
    "The staff were helpful and friendly", "A beautiful and engaging film",
    "The meal was cold and disappointing", "I hated the confusing interface",
    "Slow service and a rude manager", "The plot was boring and predictable",
    "This product broke after one day", "Waste of money and time",
    "The food was not good", "The film was not entertaining",
    "I would gladly return", "The instructions were clear and useful",
])
labels = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1])

X_train, X_test, y_train, y_test = train_test_split(
    reviews, labels, test_size=0.30, random_state=232, stratify=labels
)
sentiment_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000)),
])
sentiment_model.fit(X_train, y_train)
print(classification_report(y_test, sentiment_model.predict(X_test), target_names=["negative", "positive"]))


## Part 1 — Inspect performance

Accuracy is the fraction of correct predictions, but it can conceal which class is being missed. A confusion matrix separates true and false positive and negative predictions.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, sentiment_model.predict(X_test), display_labels=["negative", "positive"], cmap="Blues"
)

for text, actual, predicted in zip(X_test, y_test, sentiment_model.predict(X_test)):
    print(f"actual={actual}, predicted={predicted}: {text}")


### ✏️ Written response 1

Identify one prediction that would be concerning in a real product-review system. Is the problem likely caused by limited training data, the bag-of-words representation, an ambiguous label, or something else?

> **YOUR ANSWER:**


## Part 2 — What did the model learn?

For a linear model, each TF–IDF feature receives a coefficient. Large positive coefficients push a review toward the positive class; large negative coefficients push toward the negative class. These are associations in this training sample, not universal definitions of sentiment.


In [ ]:
vectorizer = sentiment_model.named_steps["tfidf"]
classifier = sentiment_model.named_steps["classifier"]
terms = vectorizer.get_feature_names_out()
weights = classifier.coef_[0]

print("Terms associated with positive sentiment:")
print(list(zip(terms[np.argsort(weights)[-8:]], np.sort(weights)[-8:].round(2))))
print("\nTerms associated with negative sentiment:")
print(list(zip(terms[np.argsort(weights)[:8]], np.sort(weights)[:8].round(2))))


### Practice and ethics check

Try the model on: “The service was not bad.” Does its result make sense? Then describe one population, dialect, or setting for which these labels and examples might be unrepresentative. What evaluation data would you need before deployment?

> **YOUR ANSWER:**


## Part 0 — From a score to a decision

Logistic regression predicts a probability between 0 and 1, then a threshold (often 0.5) turns it into a class. Text features are TF–IDF values, while sentiment labels are human measurements with uncertainty—not unquestionable facts.


In [ ]:
examples = ["excellent and useful", "cold and disappointing", "not bad"]
for review, probability in zip(examples, sentiment_model.predict_proba(examples)[:, 1]):
    print(f"{probability:.2f} positive probability — {review}")


### Prediction practice

Predict each example’s label before running the cell. If the model disagrees, consider negation, unfamiliar words, and ambiguity.

> **YOUR ANSWER:**


## Part 3 — Baselines and class balance

A classifier should outperform a sensible baseline. If 90% of reviews are positive, an always-positive classifier earns 90% accuracy while learning nothing about negative reviews. Precision asks whether positive predictions are correct; recall asks whether truly positive examples are found. The right metric depends on the cost of each error.


In [ ]:
majority_label = int(np.mean(y_train) >= 0.5)
print(f"majority-class baseline accuracy: {np.mean(y_test == majority_label):.2f}")
print(f"model accuracy: {np.mean(sentiment_model.predict(X_test) == y_test):.2f}")


### Discussion and responsibility

If negative reviews are sent to a support team, which error is worse: flagging a positive review or missing a negative review? Name one population, dialect, or setting underrepresented by this toy dataset, and the evaluation data needed before deployment.

> **YOUR ANSWER:**


## Responsible interpretation

Sentiment systems can mishandle sarcasm, quoted language, dialect, genre, and context-dependent terms. Test on representative held-out data, document annotation rules, and retain a human-review or correction route when errors can affect people.


## Part 4 — Read model coefficients carefully

Positive and negative coefficient lists show correlations in this small training sample. A high coefficient does not mean a word is always positive or negative; it may reflect topic, author style, or a coincidental pattern. Inspecting coefficients is a helpful diagnostic, not a causal explanation.


### Error-analysis protocol

For each mistake, record the review, true label, prediction, confidence, and a likely cause. Then group errors into themes—negation, ambiguity, missing vocabulary, labeling disagreement, or something else. Which theme would you address first, and how?

> **YOUR ANSWER:**


## Lesson summary

The full text-classification pipeline is: collect and label text, split data, transform text to features, train a classifier, evaluate against a baseline, and inspect errors. Every step can introduce limitations worth documenting.
